# GhostGraph-Net — CHB-MIT leave-one-subject-out experiments

This notebook reproduces every result in the manuscript *GhostGraph-Net: A Lightweight Graph-Generative Network with Ghost Convolutions for Cross-Patient EEG Seizure Detection*.

**Before you start:** `Runtime → Change runtime type → T4 GPU` (or any GPU), then `Runtime → Run all`.

What it does:
1. Mounts your Google Drive and keeps everything in `MyDrive/GhostGraph/` (so nothing is lost if Colab disconnects).
2. Downloads CHB-MIT patients chb01–chb10 from PhysioNet (seizure recordings + 3 seizure-free recordings each, ~3 GB download, stored compactly as ~1 GB).
3. Trains GG-Net, a re-implemented GGN, EEGNet, ShallowConvNet, CNN-LSTM, a static-graph GCN, an EEG Transformer and a random forest, plus four GG-Net ablations, with leakage-free leave-one-subject-out cross-validation.
4. Writes tables, statistics and figures, and packs everything into `MyDrive/GhostGraph/ghostgraph_results.zip`.

**If Colab disconnects:** reconnect and choose `Runtime → Run all` again. Finished downloads and finished runs are skipped automatically; it carries on where it stopped.

In [ ]:
import torch, subprocess, os
print("PyTorch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE")
if not torch.cuda.is_available():
    print("\n*** No GPU. Use Runtime -> Change runtime type -> T4 GPU, then Run all again. ***")

## 1. Google Drive and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
ROOT = '/content/drive/MyDrive/GhostGraph'
CODE = ROOT + '/code'; PREP = ROOT + '/chbmit_prep'; OUT = ROOT + '/results'; CACHE = '/content/cache'
for d in (CODE, PREP, OUT, CACHE): os.makedirs(d, exist_ok=True)
%env CODE=$CODE
print(ROOT)

In [ ]:
%%writefile {CODE}/data.py
"""CHB-MIT data handling for the GhostGraph study.

Reads the compact .npz files produced by prepare_chbmit.py (18 bipolar channels,
128 Hz, float16) and the PhysioNet summary files; builds 4-s windows, labels,
and leakage-free (training-fold-only) normalisation statistics.
"""
import os, re, glob
import numpy as np

FS = 128
WIN_S = 4
WIN = FS * WIN_S


def parse_summary(path):
    """Return {recording_stem: [(start_s, end_s), ...]} from a chbXX-summary.txt."""
    out, cur = {}, None
    starts = []
    with open(path, encoding="latin-1") as f:
        for line in f:
            line = line.strip()
            m = re.match(r"File Name:\s*(\S+)\.edf", line)
            if m:
                cur = m.group(1); out[cur] = []; starts = []
                continue
            if cur is None:
                continue
            m = re.match(r"Seizure(?:\s+\d+)?\s+Start Time:\s*(\d+)", line)
            if m:
                starts.append(int(m.group(1))); continue
            m = re.match(r"Seizure(?:\s+\d+)?\s+End Time:\s*(\d+)", line)
            if m and starts:
                out[cur].append((starts.pop(0), int(m.group(1))))
    return out


def load_patient(prep_dir, patient):
    """List of dicts: stem, x (C,N float32), seizures [(s,e)] for one patient."""
    summ = parse_summary(os.path.join(prep_dir, f"{patient}-summary.txt"))
    recs = []
    for f in sorted(glob.glob(os.path.join(prep_dir, f"{patient}_*.npz"))):
        stem = os.path.basename(f)[:-4]
        if stem not in summ:
            continue
        d = np.load(f)
        recs.append(dict(stem=stem, x=d["x"].astype(np.float32), seizures=summ[stem]))
    return recs


def window_starts(n_samples, stride_s):
    stride = int(stride_s * FS)
    return np.arange(0, n_samples - WIN + 1, stride)


def label_windows(starts, seizures, min_frac=0.5):
    """Window is ictal if >= min_frac of it lies inside an annotated seizure."""
    y = np.zeros(len(starts), np.int64)
    t0 = starts / FS; t1 = t0 + WIN_S
    for s, e in seizures:
        ov = np.clip(np.minimum(t1, e) - np.maximum(t0, s), 0, None)
        y[ov >= min_frac * WIN_S] = 1
    return y


def build_train_windows(recs, rng, ictal_stride=2.0, inter_stride=4.0, ratio=5,
                        exclude_margin_s=60):
    """Return X (n,C,WIN) float16, y. Ictal windows densely sampled; interictal
    windows drawn ratio:1 at random, excluding +/- margin around seizures."""
    Xi, Xn = [], []
    for r in recs:
        x, N = r["x"], r["x"].shape[1]
        st = window_starts(N, ictal_stride)
        y = label_windows(st, r["seizures"])
        for s in st[y == 1]:
            Xi.append(x[:, s:s + WIN])
        st2 = window_starts(N, inter_stride)
        keep = np.ones(len(st2), bool)
        t0 = st2 / FS
        for s, e in r["seizures"]:
            keep &= ~((t0 + WIN_S > s - exclude_margin_s) & (t0 < e + exclude_margin_s))
        Xn.extend([(r["stem"], s) for s in st2[keep]])
    n_int = min(len(Xn), max(ratio * len(Xi), 2000))
    pick = rng.choice(len(Xn), n_int, replace=False)
    lookup = {r["stem"]: r["x"] for r in recs}
    Xn = [lookup[Xn[k][0]][:, Xn[k][1]:Xn[k][1] + WIN] for k in pick]
    X = np.stack(Xi + Xn).astype(np.float16)
    y = np.r_[np.ones(len(Xi)), np.zeros(len(Xn))].astype(np.int64)
    return X, y


def build_test_windows(recs, stride_s=2.0):
    """Continuous windows over every recording (for segment + event scoring)."""
    out = []
    for r in recs:
        st = window_starts(r["x"].shape[1], stride_s)
        X = np.stack([r["x"][:, s:s + WIN] for s in st]).astype(np.float16)
        y = label_windows(st, r["seizures"])
        out.append(dict(stem=r["stem"], X=X, y=y, t=st / FS,
                        seizures=r["seizures"], dur=r["x"].shape[1] / FS))
    return out


def channel_scale(X):
    """Per-channel robust scale from TRAINING windows only."""
    Xf = X[:: max(1, len(X) // 4000)].astype(np.float32)
    med = np.median(Xf, axis=(0, 2))
    iqr = np.subtract(*np.percentile(Xf, [75, 25], axis=(0, 2)))
    return med.astype(np.float32), (iqr / 1.349 + 1e-3).astype(np.float32)


# ---------------------------------------------------------------- handcrafted
BANDS = [(0.5, 4), (4, 8), (8, 13), (13, 30), (30, 45)]
FEAT_NAMES = ["logP_delta", "logP_theta", "logP_alpha", "logP_beta", "logP_gamma",
              "log_line_length", "hjorth_mobility", "hjorth_complexity"]


def handcrafted(X):
    """X (n,C,T) -> (n,C,8): band log-power, log line length, Hjorth mob/comp."""
    X = X.astype(np.float32)
    n, C, T = X.shape
    F = np.fft.rfft(X * np.hanning(T)[None, None], axis=-1)
    P = (np.abs(F) ** 2) / T
    f = np.fft.rfftfreq(T, 1 / FS)
    feats = [np.log(P[..., (f >= lo) & (f < hi)].mean(-1) + 1e-6) for lo, hi in BANDS]
    d1 = np.diff(X, axis=-1); d2 = np.diff(d1, axis=-1)
    feats.append(np.log(np.abs(d1).sum(-1) + 1e-6))
    v0, v1, v2 = X.var(-1) + 1e-8, d1.var(-1) + 1e-8, d2.var(-1) + 1e-8
    mob = np.sqrt(v1 / v0); comp = np.sqrt(v2 / v1) / mob
    feats += [mob, comp]
    return np.stack(feats, -1).astype(np.float32)

In [ ]:
%%writefile {CODE}/models.py
"""Models for the GhostGraph seizure-detection study.

Proposed: GhostGraph-Net (GG-Net) -- a lightweight Ghost-convolution temporal
encoder, a GGN-style probabilistic connectivity-graph generator (mixture-Gaussian
edge logits + relaxed-Bernoulli/Gumbel sampling), a multi-head attentive graph
decoder, a parallel Ghost-CNN spatial branch, and interpretable spectral node
descriptors.  Baselines: EEGNet, ShallowConvNet, CNN-LSTM, static-graph GCN,
EEG Transformer, and a GGN re-implementation (standard-conv encoder).
"""
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

C_DEFAULT = 18


# ------------------------------------------------------------------ building blocks
class Ghost1d(nn.Module):
    """Ghost module (Han et al., CVPR 2020) for 1-D signals: a primary conv makes
    out/ratio intrinsic maps, a cheap depthwise conv makes the remaining 'ghost'
    maps."""

    def __init__(self, cin, cout, k, stride=1, ratio=2, dw_k=3):
        super().__init__()
        init = math.ceil(cout / ratio)
        self.cout = cout
        self.primary = nn.Sequential(nn.Conv1d(cin, init, k, stride, k // 2, bias=False),
                                     nn.BatchNorm1d(init), nn.ELU())
        self.cheap = nn.Sequential(nn.Conv1d(init, init * (ratio - 1), dw_k, 1, dw_k // 2,
                                             groups=init, bias=False),
                                   nn.BatchNorm1d(init * (ratio - 1)), nn.ELU())

    def forward(self, x):
        a = self.primary(x)
        return torch.cat([a, self.cheap(a)], 1)[:, : self.cout]


class Std1d(nn.Module):
    """Standard conv drop-in for Ghost1d (ablation)."""

    def __init__(self, cin, cout, k, stride=1, **_):
        super().__init__()
        self.f = nn.Sequential(nn.Conv1d(cin, cout, k, stride, k // 2, bias=False),
                               nn.BatchNorm1d(cout), nn.ELU())

    def forward(self, x):
        return self.f(x)


class TemporalEncoder(nn.Module):
    """Channel-shared temporal encoder: (B,C,T) -> node features (B,C,d)."""

    def __init__(self, d=48, ghost=True):
        super().__init__()
        B = Ghost1d if ghost else Std1d
        self.net = nn.Sequential(B(1, 16, 15, 4), B(16, 24, 9, 2), nn.MaxPool1d(4),
                                 B(24, 32, 7, 1), nn.MaxPool1d(2), B(32, d // 2, 5, 1))
        self.d = d

    def forward(self, x):
        b, c, t = x.shape
        h = self.net(x.reshape(b * c, 1, t))  # (BC, d/2, t')
        h = torch.cat([h.mean(-1), h.amax(-1)], 1)
        return h.reshape(b, c, self.d)


def self_loops(A):
    """Set the diagonal to 1 so that a flat (zero-variance) channel, whose
    correlation row is all zero, still has a non-zero degree."""
    eye = torch.eye(A.shape[-1], device=A.device, dtype=A.dtype)
    return A * (1 - eye) + eye


def pearson_adj(x):
    """|Pearson correlation| between channels of the raw window: (B,C,C)."""
    x = x - x.mean(-1, keepdim=True)
    x = x / (x.norm(dim=-1, keepdim=True) + 1e-6)
    return (x @ x.transpose(1, 2)).abs()


class GraphGenerator(nn.Module):
    """GGN-style connectivity-graph generator.

    For every electrode pair a Para-Learner MLP outputs the parameters of a
    K-component Gaussian mixture over the edge logit; a component is chosen with
    Gumbel-softmax, the logit is drawn by re-parameterisation, and the edge is
    sampled with a relaxed Bernoulli (binary-concrete) gate.  H heads give H
    graphs with different learned neighbourhood ranges.  At inference the
    expected edge probability is used (deterministic)."""

    def __init__(self, d, heads=3, K=3, tau=0.5, use_prior=True):
        super().__init__()
        self.H, self.K, self.tau, self.use_prior = heads, K, tau, use_prior
        ein = 2 * d + (1 if use_prior else 0)
        self.para = nn.Sequential(nn.Linear(ein, 32), nn.ELU(), nn.Linear(32, heads * K * 3))

    def forward(self, h, prior=None):
        b, c, d = h.shape
        hi = h.unsqueeze(2).expand(b, c, c, d)
        hj = h.unsqueeze(1).expand(b, c, c, d)
        e = [hi * hj, (hi - hj).abs()]
        if self.use_prior:
            e.append(prior.unsqueeze(-1))
        p = self.para(torch.cat(e, -1)).view(b, c, c, self.H, self.K, 3)
        pi_logit, mu, logsig = p[..., 0], p[..., 1], p[..., 2].clamp(-4, 2)
        if self.training:
            w = F.gumbel_softmax(pi_logit, tau=1.0, hard=False, dim=-1)
            z = (w * (mu + torch.randn_like(mu) * logsig.exp())).sum(-1)
            u = torch.rand_like(z).clamp(1e-6, 1 - 1e-6)
            A = torch.sigmoid((z + torch.log(u) - torch.log1p(-u)) / self.tau)
        else:
            w = pi_logit.softmax(-1)
            A = (w * torch.sigmoid(mu / torch.sqrt(1 + math.pi * (2 * logsig).exp() / 8))).sum(-1)
        A = A.permute(0, 3, 1, 2)  # (B,H,C,C)
        A = 0.5 * (A + A.transpose(-1, -2))
        eye = torch.eye(c, device=h.device)
        A = A * (1 - eye) + eye
        return A


class AttentiveGraphDecoder(nn.Module):
    """Graph conv over each generated graph + attention across graph heads."""

    def __init__(self, d, layers=2):
        super().__init__()
        self.W = nn.ModuleList([nn.Linear(d, d) for _ in range(layers)])
        self.att = nn.ModuleList([nn.Sequential(nn.Linear(d, 16), nn.Tanh(), nn.Linear(16, 1))
                                  for _ in range(layers)])
        self.norm = nn.ModuleList([nn.LayerNorm(d) for _ in range(layers)])
        self.last_alpha = None

    def forward(self, h, A):  # h (B,C,d); A (B,H,C,C)
        deg = A.sum(-1, keepdim=True)
        An = A / deg
        for W, att, ln in zip(self.W, self.att, self.norm):
            m = torch.einsum("bhij,bjd->bhid", An, W(h))  # (B,H,C,d)
            s = att(m.mean(2))  # (B,H,1)
            alpha = s.softmax(1).unsqueeze(-1)  # (B,H,1,1)
            self.last_alpha = alpha.detach()
            h = ln(h + F.elu((alpha * m).sum(1)))
        return h


class GhostSpatialBranch(nn.Module):
    """Parallel CNN branch: depthwise-separable spatial filter across electrodes
    then a Ghost temporal stack (global, non-graph view of the window)."""

    def __init__(self, C=C_DEFAULT, out=32, ghost=True):
        super().__init__()
        B = Ghost1d if ghost else Std1d
        self.spatial = nn.Sequential(nn.Conv1d(C, 16, 1, bias=False), nn.BatchNorm1d(16), nn.ELU())
        self.temporal = nn.Sequential(B(16, 24, 15, 2), nn.MaxPool1d(4), B(24, out, 7, 1),
                                      nn.AdaptiveAvgPool1d(1))

    def forward(self, x):
        return self.temporal(self.spatial(x)).squeeze(-1)


class GhostGraphNet(nn.Module):
    """Proposed GhostGraph-Net (GG-Net)."""

    def __init__(self, C=C_DEFAULT, nfeat=8, d=48, heads=3, K=3, ghost=True,
                 dynamic_graph=True, handcrafted=True, cnn_branch=True):
        super().__init__()
        self.handcrafted, self.cnn_branch, self.dynamic = handcrafted, cnn_branch, dynamic_graph
        self.enc = TemporalEncoder(d, ghost)
        self.feat_proj = nn.Linear(nfeat, d) if handcrafted else None
        self.gen = GraphGenerator(d, heads if dynamic_graph else 1, K) if dynamic_graph else None
        self.dec = AttentiveGraphDecoder(d)
        self.cnn = GhostSpatialBranch(C, 32, ghost) if cnn_branch else None
        fin = 2 * d + (32 if cnn_branch else 0)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(fin, 32), nn.ELU(), nn.Linear(32, 2))
        self.last_A = None

    def forward(self, x, f=None):
        h = self.enc(x)
        if self.handcrafted:
            h = h + self.feat_proj(f)
        prior = pearson_adj(x)
        A = self.gen(h, prior) if self.dynamic else self_loops(prior).unsqueeze(1)
        self.last_A = A.detach()
        h = self.dec(h, A)
        z = [h.mean(1), h.amax(1)]
        if self.cnn_branch:
            z.append(self.cnn(x))
        return self.head(torch.cat(z, 1))


class GGNReimpl(nn.Module):
    """Re-implementation of the GGN of Li et al. (Sci. Rep. 2022) adapted to binary
    detection: standard-conv temporal encoder, graph generator, attentive graph
    decoder, shallow CNN branch on temporal features; no Ghost modules and no
    handcrafted features."""

    def __init__(self, C=C_DEFAULT, d=64, heads=3, K=3):
        super().__init__()
        self.enc = TemporalEncoder(d, ghost=False)
        self.gen = GraphGenerator(d, heads, K, use_prior=True)
        self.dec = AttentiveGraphDecoder(d)
        self.cnn = nn.Sequential(nn.Conv1d(C, 32, 3, padding=1), nn.ELU(), nn.AdaptiveAvgPool1d(1))
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * d + 32, 32), nn.ELU(), nn.Linear(32, 2))

    def forward(self, x, f=None):
        h0 = self.enc(x)
        A = self.gen(h0, pearson_adj(x))
        h = self.dec(h0, A)
        c = self.cnn(h0).squeeze(-1)
        return self.head(torch.cat([h.mean(1), h.amax(1), c], 1))


# ------------------------------------------------------------------------ baselines
class EEGNet(nn.Module):
    """EEGNet-8,2 (Lawhern et al., J. Neural Eng. 2018), kernel scaled to 128 Hz."""

    def __init__(self, C=C_DEFAULT, T=512, F1=8, D=2, F2=16, k=64, p=0.25):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(1, F1, (1, k), padding=(0, k // 2), bias=False),
                                nn.BatchNorm2d(F1),
                                nn.Conv2d(F1, F1 * D, (C, 1), groups=F1, bias=False),
                                nn.BatchNorm2d(F1 * D), nn.ELU(), nn.AvgPool2d((1, 4)), nn.Dropout(p))
        self.b2 = nn.Sequential(nn.Conv2d(F1 * D, F1 * D, (1, 16), padding=(0, 8), groups=F1 * D, bias=False),
                                nn.Conv2d(F1 * D, F2, 1, bias=False), nn.BatchNorm2d(F2), nn.ELU(),
                                nn.AvgPool2d((1, 8)), nn.Dropout(p))
        with torch.no_grad():
            n = self.b2(self.b1(torch.zeros(1, 1, C, T))).numel()
        self.fc = nn.Linear(n, 2)

    def forward(self, x, f=None):
        return self.fc(self.b2(self.b1(x.unsqueeze(1))).flatten(1))


class ShallowConvNet(nn.Module):
    """ShallowFBCSPNet (Schirrmeister et al., HBM 2017) at 128 Hz."""

    def __init__(self, C=C_DEFAULT, T=512, nf=40):
        super().__init__()
        self.c1 = nn.Conv2d(1, nf, (1, 13))
        self.c2 = nn.Conv2d(nf, nf, (C, 1), bias=False)
        self.bn = nn.BatchNorm2d(nf)
        self.pool = nn.AvgPool2d((1, 37), (1, 7))
        self.drop = nn.Dropout(0.5)
        with torch.no_grad():
            n = self._feat(torch.zeros(1, C, T)).numel()
        self.fc = nn.Linear(n, 2)

    def _feat(self, x):
        z = self.bn(self.c2(self.c1(x.unsqueeze(1))))
        return torch.log(torch.clamp(self.pool(z * z), 1e-6)).flatten(1)

    def forward(self, x, f=None):
        return self.fc(self.drop(self._feat(x)))


class CNNLSTM(nn.Module):
    """1-D CNN front-end + bidirectional LSTM over time."""

    def __init__(self, C=C_DEFAULT):
        super().__init__()
        self.cnn = nn.Sequential(nn.Conv1d(C, 32, 7, 2, 3), nn.BatchNorm1d(32), nn.ReLU(), nn.MaxPool1d(2),
                                 nn.Conv1d(32, 64, 5, 1, 2), nn.BatchNorm1d(64), nn.ReLU(), nn.MaxPool1d(4))
        self.lstm = nn.LSTM(64, 48, batch_first=True, bidirectional=True)
        self.fc = nn.Sequential(nn.Dropout(0.3), nn.Linear(96, 2))

    def forward(self, x, f=None):
        z, _ = self.lstm(self.cnn(x).transpose(1, 2))
        return self.fc(z.mean(1))


class StaticGCN(nn.Module):
    """GCN on a fixed correlation graph (static-topology GNN baseline)."""

    def __init__(self, C=C_DEFAULT, nfeat=8, d=48):
        super().__init__()
        self.enc = TemporalEncoder(d, ghost=False)
        self.W1, self.W2 = nn.Linear(d, d), nn.Linear(d, d)
        self.head = nn.Sequential(nn.Dropout(0.3), nn.Linear(2 * d, 2))

    def forward(self, x, f=None):
        A = self_loops(pearson_adj(x))
        An = A / A.sum(-1, keepdim=True)
        h = self.enc(x)
        h = F.elu(An @ self.W1(h))
        h = F.elu(An @ self.W2(h))
        return self.head(torch.cat([h.mean(1), h.amax(1)], 1))


class EEGTransformer(nn.Module):
    """Compact EEG Transformer: conv patch embedding (0.25 s patches) + 2 encoder
    layers + CLS token (in the spirit of EEG Conformer, Song et al. 2023)."""

    def __init__(self, C=C_DEFAULT, T=512, d=64, patch=32):
        super().__init__()
        self.embed = nn.Sequential(nn.Conv1d(C, d, patch, patch), nn.BatchNorm1d(d))
        n = T // patch
        self.cls = nn.Parameter(torch.zeros(1, 1, d))
        self.pos = nn.Parameter(torch.randn(1, n + 1, d) * 0.02)
        layer = nn.TransformerEncoderLayer(d, 4, 128, 0.2, batch_first=True, norm_first=True)
        self.tf = nn.TransformerEncoder(layer, 2)
        self.fc = nn.Linear(d, 2)

    def forward(self, x, f=None):
        z = self.embed(x).transpose(1, 2)
        z = torch.cat([self.cls.expand(len(z), -1, -1), z], 1) + self.pos
        return self.fc(self.tf(z)[:, 0])


def build(name, C=C_DEFAULT):
    return {
        "GG-Net": lambda: GhostGraphNet(C),
        "GG-Net w/o graph generator": lambda: GhostGraphNet(C, dynamic_graph=False),
        "GG-Net w/o Ghost (std conv)": lambda: GhostGraphNet(C, ghost=False),
        "GG-Net w/o handcrafted": lambda: GhostGraphNet(C, handcrafted=False),
        "GG-Net w/o CNN branch": lambda: GhostGraphNet(C, cnn_branch=False),
        "GGN (re-impl.)": lambda: GGNReimpl(C),
        "EEGNet": lambda: EEGNet(C),
        "ShallowConvNet": lambda: ShallowConvNet(C),
        "CNN-LSTM": lambda: CNNLSTM(C),
        "Static GCN": lambda: StaticGCN(C),
        "EEG Transformer": lambda: EEGTransformer(C),
    }[name]()


def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

In [ ]:
%%writefile {CODE}/events.py
"""Event-level scoring following the SzCORE conventions (Dan et al., Epilepsia 2024):
any-overlap matching with 30 s pre-onset and 60 s post-offset tolerance,
hypothesis events closer than 90 s merged, events longer than 5 min split."""
import numpy as np

STRIDE = 2.0      # s between window starts
WIN = 4.0         # s window length
PRE, POST, MERGE, MAXLEN = 30.0, 60.0, 90.0, 300.0


def smooth(p, k=3):
    """Causal moving average over k consecutive windows (k*2 s)."""
    p = np.asarray(p, np.float64)
    c = np.cumsum(np.r_[0.0, p])
    i = np.arange(len(p))
    lo = np.maximum(0, i - k + 1)
    return (c[i + 1] - c[lo]) / (i + 1 - lo)


def to_events(binary, t):
    ev, on = [], None
    for b, s in zip(binary, t):
        if b and on is None:
            on = s
        if not b and on is not None:
            ev.append([on, prev + WIN]); on = None
        prev = s
    if on is not None:
        ev.append([on, t[-1] + WIN])
    merged = []
    for e in ev:
        if merged and e[0] - merged[-1][1] < MERGE:
            merged[-1][1] = e[1]
        else:
            merged.append(e)
    out = []
    for s, e in merged:
        while e - s > MAXLEN:
            out.append([s, s + MAXLEN]); s += MAXLEN
        out.append([s, e])
    return out


def event_scores(rec, binary):
    hyp = to_events(np.asarray(binary, bool), rec["t"])
    ref = rec["seizures"]
    tp, lat, used = 0, [], np.zeros(len(hyp), bool)
    for s, e in ref:
        hit = [k for k, (hs, he) in enumerate(hyp) if he >= s - PRE and hs <= e + POST]
        if hit:
            tp += 1; used[hit] = True
            lat.append(max(0.0, hyp[hit[0]][0] + WIN - s))   # decision available at window end
    fp = int((~used).sum())
    return dict(tp=tp, n_sz=len(ref), fp=fp, latency=lat)

In [ ]:
%%writefile {CODE}/run_loso.py
"""Leave-one-subject-out experiments on CHB-MIT for the GhostGraph study.

usage: python run_loso.py --prep DIR --out DIR --models "GG-Net,EEGNet" [--epochs 15] [--seeds 0,1,2]
Every (model, test patient, seed) result is saved as JSON + NPZ, so runs resume
after an interruption. Uses the GPU when one is available.
"""
import os, sys, json, time, argparse
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.ensemble import RandomForestClassifier

sys.path.insert(0, os.path.dirname(os.path.abspath(__file__)))
import data as D
import models as M
from events import event_scores, smooth

DEV = torch.device("cuda" if torch.cuda.is_available() else "cpu")
if DEV.type == "cpu":
    torch.set_num_threads(max(1, os.cpu_count()))
else:
    torch.backends.cudnn.benchmark = True


# --------------------------------------------------------------------- caching
def patient_cache(prep, cache, p, seed=0):
    f = os.path.join(cache, f"{p}_train.npz")
    if os.path.exists(f):
        d = np.load(f)
        return d["X"], d["y"], d["F"]
    recs = D.load_patient(prep, p)
    X, y = D.build_train_windows(recs, np.random.default_rng(seed))
    Fh = D.handcrafted(X)
    np.savez(f, X=X, y=y, F=Fh)
    return X, y, Fh


def patient_test(prep, cache, p):
    f = os.path.join(cache, f"{p}_test.npz")
    if os.path.exists(f):
        d = np.load(f, allow_pickle=True)
        return list(d["T"])
    recs = D.load_patient(prep, p)
    T = D.build_test_windows(recs)
    for r in T:
        r["F"] = D.handcrafted(r["X"])
    np.savez(f, T=np.array(T, dtype=object))
    return T


# --------------------------------------------------------------------- training
def norm(X, med, sc):
    return np.clip((X.astype(np.float32) - med[None, :, None]) / sc[None, :, None], -20, 20)


def predict(model, X, Fh, med, sc, fmu, fsd, bs=1024, graphs=False):
    model.eval(); out, As = [], []
    with torch.no_grad():
        for i in range(0, len(X), bs):
            x = torch.from_numpy(norm(X[i:i + bs], med, sc)).to(DEV)
            f = torch.from_numpy(((Fh[i:i + bs] - fmu) / fsd).astype(np.float32)).to(DEV)
            out.append(model(x, f).softmax(-1)[:, 1].float().cpu().numpy())
            if graphs:
                As.append(model.last_A.float().cpu().numpy().astype(np.float16))
    p = np.concatenate(out) if out else np.zeros(0)
    return (p, np.concatenate(As)) if graphs else p


def train_nn(name, Xtr, ytr, Ftr, Xva, yva, Fva, med, sc, fmu, fsd, epochs, seed):
    torch.manual_seed(seed); np.random.seed(seed); rng = np.random.default_rng(seed)
    model = M.build(name, Xtr.shape[1]).to(DEV)
    opt = torch.optim.AdamW(model.parameters(), 1e-3, weight_decay=1e-3)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    w = torch.tensor([1.0, (ytr == 0).sum() / max(1, (ytr == 1).sum())], dtype=torch.float32, device=DEV)
    best, best_state, bs = -1, None, 128
    hist = []
    for ep in range(epochs):
        model.train(); idx = rng.permutation(len(Xtr)); tl = 0
        for i in range(0, len(idx), bs):
            j = np.sort(idx[i:i + bs])
            x = norm(Xtr[j], med, sc)
            x *= rng.uniform(0.8, 1.2, (len(j), 1, 1)).astype(np.float32)          # amplitude scaling
            x += rng.normal(0, 0.05, x.shape).astype(np.float32)                  # additive noise
            sh = rng.integers(-13, 14); x = np.roll(x, sh, -1)                     # +-100 ms shift
            f = ((Ftr[j] - fmu) / fsd).astype(np.float32)
            logit = model(torch.from_numpy(x).to(DEV), torch.from_numpy(f).to(DEV))
            loss = F.cross_entropy(logit, torch.from_numpy(ytr[j]).to(DEV), weight=w)
            opt.zero_grad(); loss.backward(); nn.utils.clip_grad_norm_(model.parameters(), 5.0); opt.step()
            tl += loss.item() * len(j)
        sched.step()
        pv = predict(model, Xva, Fva, med, sc, fmu, fsd)
        auc = roc_auc_score(yva, pv) if len(set(yva)) > 1 else 0.5
        hist.append(dict(epoch=ep, loss=tl / len(Xtr), val_auc=float(auc)))
        if auc > best:
            best = auc; best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
    model.load_state_dict(best_state)
    return model, hist


# --------------------------------------------------------------------- metrics
def choose_threshold(val_tests, probs):
    """Threshold maximising event-level F1 on the VALIDATION patient (never the
    test patient); ties broken toward the higher threshold (fewer false alarms)."""
    best, thr = -1, 0.5
    for t in np.arange(0.30, 0.96, 0.05):
        res = [event_scores(r, smooth(p) >= t) for r, p in zip(val_tests, probs)]
        tp = sum(x["tp"] for x in res); n = sum(x["n_sz"] for x in res); fp = sum(x["fp"] for x in res)
        sen = tp / max(1, n); prec = tp / max(1, tp + fp)
        f1 = 2 * sen * prec / max(1e-9, sen + prec)
        if f1 >= best:
            best, thr = f1, t
    return float(thr)


def segment_metrics(y, p, thr):
    yh = (smooth(p) >= thr).astype(int)
    tp = int(((yh == 1) & (y == 1)).sum()); tn = int(((yh == 0) & (y == 0)).sum())
    fp = int(((yh == 1) & (y == 0)).sum()); fn = int(((yh == 0) & (y == 1)).sum())
    sen = tp / max(1, tp + fn); spe = tn / max(1, tn + fp); prec = tp / max(1, tp + fp)
    return dict(acc=(tp + tn) / len(y), sen=sen, spe=spe, prec=prec,
                f1=2 * sen * prec / max(1e-9, sen + prec), bacc=(sen + spe) / 2,
                auroc=float(roc_auc_score(y, p)) if len(set(y)) > 1 else float("nan"),
                auprc=float(average_precision_score(y, p)) if len(set(y)) > 1 else float("nan"),
                tp=tp, tn=tn, fp=fp, fn=fn)


def latency_ms(model, X, Fh, med, sc, fmu, fsd):
    xb = torch.from_numpy(norm(X[:1], med, sc)).to(DEV)
    fb = torch.from_numpy(((Fh[:1] - fmu) / fsd).astype(np.float32)).to(DEV)
    model.eval()
    with torch.no_grad():
        for _ in range(10): model(xb, fb)
        if DEV.type == "cuda": torch.cuda.synchronize()
        ts = time.perf_counter()
        for _ in range(100): model(xb, fb)
        if DEV.type == "cuda": torch.cuda.synchronize()
    return (time.perf_counter() - ts) / 100 * 1000


# --------------------------------------------------------------------- main
def run_one(a, name, k, P, seed, cache):
    test_p = P[k]
    tag = f"{name}__{test_p}__s{seed}".replace(" ", "_").replace("/", "-").replace(".", "")
    fout = os.path.join(a.out, "runs", tag + ".json")
    if os.path.exists(fout):
        return
    val_p = P[(k + 1) % len(P)]
    tr_p = [q for q in P if q not in (test_p, val_p)]
    t0 = time.time()
    parts = [patient_cache(a.prep, cache, q) for q in tr_p]
    Xtr = np.concatenate([x[0] for x in parts]); ytr = np.concatenate([x[1] for x in parts])
    Ftr = np.concatenate([x[2] for x in parts]); del parts
    Xva, yva, Fva = patient_cache(a.prep, cache, val_p)
    med, sc = D.channel_scale(Xtr)
    fmu = Ftr.mean((0, 1), keepdims=True)[0]; fsd = Ftr.std((0, 1), keepdims=True)[0] + 1e-6
    val_T = patient_test(a.prep, cache, val_p); test_T = patient_test(a.prep, cache, test_p)
    hist, params, lat, graphs = [], None, None, None
    if name == "RF (handcrafted)":
        rf = RandomForestClassifier(300, min_samples_leaf=2, class_weight="balanced", n_jobs=-1, random_state=seed)
        rf.fit(Ftr.reshape(len(Ftr), -1), ytr)
        pr = lambda T: [rf.predict_proba(r["F"].reshape(len(r["F"]), -1))[:, 1] for r in T]
        pt = pr(test_T)
    else:
        model, hist = train_nn(name, Xtr, ytr, Ftr, Xva, yva, Fva, med, sc, fmu, fsd, a.epochs, seed)
        pr = lambda T: [predict(model, r["X"], r["F"], med, sc, fmu, fsd) for r in T]
        params = M.n_params(model)
        lat = latency_ms(model, test_T[0]["X"], test_T[0]["F"], med, sc, fmu, fsd)
        if name == "GG-Net" and seed == 0:
            pt, graphs = [], {}
            for r in test_T:
                p_, A_ = predict(model, r["X"], r["F"], med, sc, fmu, fsd, graphs=True)
                pt.append(p_)
                for s_i, (s, e) in enumerate(r["seizures"]):
                    keep = (r["t"] >= s - 120) & (r["t"] <= e + 60)
                    graphs[f"{r['stem']}_sz{s_i}"] = dict(A=A_[keep], t=r["t"][keep], onset=s, offset=e)
            alpha = getattr(model.dec, "last_alpha", None)
        else:
            pt = pr(test_T)
    thr = choose_threshold(val_T, pr(val_T))
    y = np.concatenate([r["y"] for r in test_T]); p = np.concatenate(pt)
    seg = segment_metrics(y, p, thr)
    ev = [event_scores(r, smooth(q) >= thr) for r, q in zip(test_T, pt)]
    hours = sum(r["dur"] for r in test_T) / 3600
    tp = sum(e["tp"] for e in ev); nsz = sum(e["n_sz"] for e in ev); fp = sum(e["fp"] for e in ev)
    lats = [float(l) for e in ev for l in e["latency"]]
    evm = dict(n_sz=nsz, tp=tp, fp=fp, hours=hours, ev_sen=tp / max(1, nsz), ev_prec=tp / max(1, tp + fp),
               fp_h=fp / hours, latency_mean=float(np.mean(lats)) if lats else None,
               latency_median=float(np.median(lats)) if lats else None, latencies=lats)
    evm["ev_f1"] = 2 * evm["ev_sen"] * evm["ev_prec"] / max(1e-9, evm["ev_sen"] + evm["ev_prec"])
    res = dict(model=name, test=test_p, val=val_p, seed=seed, thr=thr, params=params, latency_ms=lat,
               device=str(DEV), epochs=a.epochs, train_n=int(len(ytr)), train_pos=int(ytr.sum()),
               seg=seg, event=evm, hist=hist, minutes=(time.time() - t0) / 60)
    np.savez_compressed(os.path.join(a.out, "runs", tag + "_pred.npz"), p=p.astype(np.float16), y=y.astype(np.int8),
                        stems=np.array([r["stem"] for r in test_T]), lens=np.array([len(r["y"]) for r in test_T]))
    if graphs:
        np.savez_compressed(os.path.join(a.out, "runs", tag + "_graphs.npz"),
                            **{f"{k}__{f}": v[f] for k, v in graphs.items() for f in ("A", "t", "onset", "offset")})
    json.dump(res, open(fout, "w"), indent=1)
    print(f"{name:28s} {test_p} s{seed} AUROC={seg['auroc']:.3f} sen={seg['sen']:.3f} spe={seg['spe']:.3f} "
          f"evSen={evm['ev_sen']:.2f} FP/h={evm['fp_h']:.2f} ({res['minutes']:.1f} min)", flush=True)


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--prep", required=True); ap.add_argument("--out", required=True)
    ap.add_argument("--cache", default=None, help="fast local dir for window caches (default OUT/cache)")
    ap.add_argument("--models", required=True); ap.add_argument("--epochs", type=int, default=15)
    ap.add_argument("--patients", default=",".join(f"chb{i:02d}" for i in range(1, 11)))
    ap.add_argument("--seeds", default="0")
    a = ap.parse_args()
    P = a.patients.split(","); cache = a.cache or os.path.join(a.out, "cache")
    os.makedirs(cache, exist_ok=True); os.makedirs(os.path.join(a.out, "runs"), exist_ok=True)
    print("device:", DEV, flush=True)
    for seed in [int(s) for s in a.seeds.split(",")]:
        for name in a.models.split(","):
            for k in range(len(P)):
                run_one(a, name, k, P, seed, cache)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/analyze.py
"""Aggregate LOSO runs into manuscript tables, statistics and figures.

usage: python analyze.py --runs RESULTS/runs --out RESULTS/analysis
Writes: summary_tables.md, per_patient.csv, stats.json, fig2_roc.png,
fig3_graphs.png (if GG-Net graphs exist), fig4_per_patient.png
"""
import os, glob, json, argparse
import numpy as np
from scipy import stats
from sklearn.metrics import roc_curve, roc_auc_score
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MAIN = ["RF (handcrafted)", "EEGNet", "ShallowConvNet", "CNN-LSTM", "Static GCN", "EEG Transformer", "GGN (re-impl.)", "GG-Net"]
ABL = ["GG-Net", "GG-Net w/o graph generator", "GG-Net w/o Ghost (std conv)", "GG-Net w/o handcrafted", "GG-Net w/o CNN branch"]
LABEL = {"RF (handcrafted)": "RF (spectral descriptors)", "EEGNet": "EEGNet-8,2", "Static GCN": "Static-graph GCN",
         "GGN (re-impl.)": "GGN (re-implemented)", "GG-Net": "GG-Net (proposed)"}
COLORS = {"GG-Net": "#B05A2A", "GGN (re-impl.)": "#3B5BA5", "EEGNet": "#2E7D6B", "EEG Transformer": "#7A4E9C",
          "Static GCN": "#B07D1A", "CNN-LSTM": "#5B6470", "ShallowConvNet": "#9AA3AD", "RF (handcrafted)": "#C9CED6"}


def load(runs):
    R = {}
    for f in glob.glob(os.path.join(runs, "*.json")):
        d = json.load(open(f))
        R.setdefault(d["model"], {}).setdefault(d["test"], []).append(d)
    return R


def per_patient(R, m, key):
    """mean over seeds for each test patient."""
    out = {}
    for p, lst in R.get(m, {}).items():
        vals = [key(d) for d in lst]
        vals = [v for v in vals if v is not None and not (isinstance(v, float) and np.isnan(v))]
        if vals:
            out[p] = float(np.mean(vals))
    return out


METRICS = {
    "auroc": lambda d: d["seg"]["auroc"], "auprc": lambda d: d["seg"]["auprc"],
    "sen": lambda d: 100 * d["seg"]["sen"], "spe": lambda d: 100 * d["seg"]["spe"], "f1": lambda d: 100 * d["seg"]["f1"],
    "ev_sen": lambda d: 100 * d["event"]["ev_sen"], "ev_prec": lambda d: 100 * d["event"]["ev_prec"],
    "ev_f1": lambda d: 100 * d["event"]["ev_f1"], "fp_h": lambda d: d["event"]["fp_h"],
    "lat_med": lambda d: d["event"]["latency_median"],
}


def ms(v, fmt):
    v = np.array(list(v.values()))
    return (fmt % v.mean()) + " ± " + (fmt % v.std(ddof=1) if len(v) > 1 else "0")


def holm(pvals):
    order = np.argsort(pvals); m = len(pvals); adj = np.empty(m); run = 0
    for r, i in enumerate(order):
        run = max(run, (m - r) * pvals[i]); adj[i] = min(1.0, run)
    return adj


def main():
    ap = argparse.ArgumentParser(); ap.add_argument("--runs", required=True); ap.add_argument("--out", required=True)
    a = ap.parse_args(); os.makedirs(a.out, exist_ok=True)
    R = load(a.runs)
    models = [m for m in MAIN if m in R]; abl = [m for m in ABL if m in R]
    patients = sorted({p for m in R for p in R[m]})
    PP = {m: {k: per_patient(R, m, f) for k, f in METRICS.items()} for m in R}
    seeds = {m: sorted({d["seed"] for lst in R[m].values() for d in lst}) for m in R}
    L = ["# GhostGraph LOSO results", "", f"Patients: {', '.join(patients)}", ""]
    L += ["Seeds per model: " + "; ".join(f"{m}: {seeds[m]}" for m in R), ""]

    # ---- window level
    L += ["## Table 3 - window level (mean ± SD over patients)", "", "| Model | AUROC | AUPRC | Sen (%) | Spe (%) | F1 (%) |", "|---|---|---|---|---|---|"]
    for m in models:
        P = PP[m]
        L.append(f"| {LABEL.get(m, m)} | {ms(P['auroc'], '%.3f')} | {ms(P['auprc'], '%.3f')} | {ms(P['sen'], '%.1f')} | {ms(P['spe'], '%.1f')} | {ms(P['f1'], '%.1f')} |")
    # ---- event level
    L += ["", "## Table 4 - event level (SzCORE)", "", "| Model | Detected/total | Sen (%) | Prec (%) | F1 (%) | FP/h | Latency median [IQR] (s) |", "|---|---|---|---|---|---|---|"]
    for m in models:
        P = PP[m]
        s0 = [d for lst in R[m].values() for d in lst if d["seed"] == seeds[m][0]]
        det = sum(d["event"]["tp"] for d in s0); tot = sum(d["event"]["n_sz"] for d in s0)
        lats = [l for d in s0 for l in d["event"]["latencies"]]
        lq = f"{np.median(lats):.0f} [{np.percentile(lats, 25):.0f}–{np.percentile(lats, 75):.0f}]" if lats else "—"
        L.append(f"| {LABEL.get(m, m)} | {det}/{tot} | {ms(P['ev_sen'], '%.1f')} | {ms(P['ev_prec'], '%.1f')} | {ms(P['ev_f1'], '%.1f')} | {ms(P['fp_h'], '%.2f')} | {lq} |")

    # ---- statistics
    st = {}
    for metric in ["auroc", "ev_f1", "fp_h"]:
        common = [p for p in patients if all(p in PP[m][metric] for m in models)]
        if len(models) >= 3 and len(common) >= 5:
            M_ = np.array([[PP[m][metric][p] for m in models] for p in common])
            chi, pf = stats.friedmanchisquare(*M_.T)
            raw, names = [], []
            for j, m in enumerate(models):
                if m == "GG-Net": continue
                try:
                    raw.append(stats.wilcoxon(M_[:, models.index("GG-Net")], M_[:, j], zero_method="zsplit").pvalue)
                except ValueError:
                    raw.append(1.0)
                names.append(m)
            st[metric] = dict(n=len(common), friedman_chi2=float(chi), friedman_p=float(pf),
                              wilcoxon_raw=dict(zip(names, map(float, raw))), wilcoxon_holm=dict(zip(names, map(float, holm(np.array(raw))))))
    json.dump(st, open(os.path.join(a.out, "stats.json"), "w"), indent=1)
    L += ["", "## Statistics (Friedman; Wilcoxon GG-Net vs each, Holm)", "", "```", json.dumps(st, indent=1), "```"]

    # ---- ablation
    if abl:
        L += ["", "## Table 5 - ablation", "", "| Variant | Params | AUROC | Event F1 (%) | Δ F1 | p (Wilcoxon vs full) |", "|---|---|---|---|---|---|"]
        full = PP.get("GG-Net", {}).get("ev_f1", {})
        for m in abl:
            prm = next((d["params"] for lst in R[m].values() for d in lst), None)
            f1 = PP[m]["ev_f1"]; common = [p for p in f1 if p in full]
            d = np.mean([f1[p] - full[p] for p in common]) if common else float("nan")
            try:
                pv = stats.wilcoxon([f1[p] for p in common], [full[p] for p in common], zero_method="zsplit").pvalue if m != "GG-Net" else None
            except ValueError:
                pv = 1.0
            L.append(f"| {m} | {prm} | {np.mean(list(PP[m]['auroc'].values())):.3f} | {np.mean(list(f1.values())):.1f} | "
                     f"{'—' if m == 'GG-Net' else f'{d:+.1f}'} | {'—' if pv is None else f'{pv:.3f}'} |")

    # ---- per patient (GG-Net)
    if "GG-Net" in R:
        L += ["", "## Table 6 - GG-Net per patient (seed 0)", "", "| Test | Val | AUROC | Detected/total | Event F1 (%) | FP/h | Median latency (s) |", "|---|---|---|---|---|---|---|"]
        rows = ["test,val,auroc,tp,n_sz,ev_f1,fp_h,lat_med,thr"]
        for p in patients:
            d = next((x for x in R["GG-Net"].get(p, []) if x["seed"] == 0), None)
            if not d: continue
            e = d["event"]
            lm = "—" if e["latency_median"] is None else "%.0f" % e["latency_median"]
            L.append(f"| {p} | {d['val']} | {d['seg']['auroc']:.3f} | {e['tp']}/{e['n_sz']} | {100*e['ev_f1']:.1f} | {e['fp_h']:.2f} | {lm} |")
            rows.append(f"{p},{d['val']},{d['seg']['auroc']:.4f},{e['tp']},{e['n_sz']},{e['ev_f1']:.4f},{e['fp_h']:.4f},{e['latency_median']},{d['thr']}")
        open(os.path.join(a.out, "per_patient.csv"), "w").write("\n".join(rows))
    open(os.path.join(a.out, "summary_tables.md"), "w").write("\n".join(L))

    plt.rcParams.update({"font.family": "DejaVu Sans", "font.size": 8})
    # ---- Fig 2 ROC pooled
    fig, ax = plt.subplots(figsize=(3.6, 3.4), dpi=300)
    for m in models:
        ys, ps = [], []
        for f in glob.glob(os.path.join(a.runs, m.replace(" ", "_").replace("/", "-").replace(".", "") + "__chb*__s0_pred.npz")):
            d = np.load(f); ys.append(d["y"]); ps.append(d["p"].astype(np.float32))
        if not ys: continue
        y, p = np.concatenate(ys), np.concatenate(ps)
        fpr, tpr, _ = roc_curve(y, p)
        ax.plot(fpr, tpr, lw=1.6 if m == "GG-Net" else 0.9, color=COLORS.get(m), label=f"{LABEL.get(m, m)} ({roc_auc_score(y, p):.3f})")
    ax.plot([0, 1], [0, 1], ls="--", lw=0.6, color="#999")
    ax.set_xlabel("False positive rate"); ax.set_ylabel("True positive rate")
    ax.legend(fontsize=5.5, frameon=False, loc="lower right"); ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(os.path.join(a.out, "fig2_roc.png")); plt.close(fig)

    # ---- Fig 4 per-patient event F1 and FP/h
    fig, axs = plt.subplots(1, 2, figsize=(7.0, 2.8), dpi=300)
    for ax, key, lab in [(axs[0], "ev_f1", "Event F1 (%)"), (axs[1], "fp_h", "False positives per hour")]:
        data = [list(PP[m][key].values()) for m in models]
        bp = ax.boxplot(data, patch_artist=True, widths=0.6, medianprops=dict(color="black", lw=0.8))
        for b, m in zip(bp["boxes"], models):
            b.set_facecolor(COLORS.get(m, "#ccc")); b.set_alpha(0.75); b.set_linewidth(0.5)
        for i, d in enumerate(data):
            ax.scatter(np.full(len(d), i + 1) + np.random.default_rng(i).uniform(-0.15, 0.15, len(d)), d, s=5, color="black", zorder=3)
        ax.set_xticks(range(1, len(models) + 1)); ax.set_xticklabels([LABEL.get(m, m).replace(" (", "\n(") for m in models], rotation=45, ha="right", fontsize=6)
        ax.set_ylabel(lab); ax.spines[["top", "right"]].set_visible(False)
    fig.tight_layout(); fig.savefig(os.path.join(a.out, "fig4_per_patient.png")); plt.close(fig)

    # ---- Fig 3 learned graphs (GG-Net, best-detected seizure)
    gfiles = sorted(glob.glob(os.path.join(a.runs, "GG-Net__chb*__s0_graphs.npz")))
    if gfiles:
        best = None
        for f in gfiles:
            g = np.load(f); keys = sorted({k.split("__")[0] for k in g.keys()})
            for k in keys:
                A, t, on, off = g[k + "__A"].astype(np.float32), g[k + "__t"], float(g[k + "__onset"]), float(g[k + "__offset"])
                if len(t) < 10: continue
                score = off - on
                if best is None or score > best[0]:
                    best = (score, os.path.basename(f).split("__")[1], k, A, t, on, off)
        if best:
            _, pat, k, A, t, on, off = best
            phases = [("interictal", (t + 4 <= on - 30)), ("onset", (t >= on - 2) & (t <= on + 10)), ("ictal", (t > on + 10) & (t + 4 <= off))]
            fig, axs = plt.subplots(3, 3, figsize=(6.0, 6.0), dpi=300)
            for c, (nm, msk) in enumerate(phases):
                if msk.sum() == 0: continue
                Am = A[msk].mean(0)
                for h in range(3):
                    im = axs[h, c].imshow(Am[h], vmin=0, vmax=1, cmap="Oranges")
                    axs[h, c].set_xticks([]); axs[h, c].set_yticks([])
                    if h == 0: axs[h, c].set_title(f"{nm} (n={int(msk.sum())})", fontsize=8)
                    if c == 0: axs[h, c].set_ylabel(f"head {h+1}", fontsize=8)
            fig.suptitle(f"GG-Net expected connectivity, {pat} {k} (onset {on:.0f} s)", fontsize=8)
            fig.colorbar(im, ax=axs, shrink=0.6, label="edge probability")
            fig.savefig(os.path.join(a.out, "fig3_graphs.png")); plt.close(fig)
    print("\n".join(L))


if __name__ == "__main__":
    main()

In [ ]:
%%writefile {CODE}/prepare_chbmit.py
"""Download and compact the CHB-MIT subset (chb01-chb10) for the GhostGraph study.

usage: python prepare_chbmit.py --out /content/drive/MyDrive/GhostGraph/chbmit_prep [--workers 6]
Keeps every recording with a seizure plus the first 3 seizure-free recordings
per patient, the 18 common bipolar channels, 256 -> 128 Hz (windowed-sinc
low-pass at 50 Hz), float16 .npz.  Safe to re-run: finished files are skipped.
"""
import os, time, argparse, urllib.request, tempfile
from concurrent.futures import ThreadPoolExecutor, as_completed
import numpy as np

BASE = "https://physionet.org/files/chbmit/1.0.0/"
CHANNELS = ["FP1-F7", "F7-T7", "T7-P7", "P7-O1", "FP1-F3", "F3-C3", "C3-P3", "P3-O1",
            "FP2-F4", "F4-C4", "C4-P4", "P4-O2", "FP2-F8", "F8-T8", "T8-P8", "P8-O2",
            "FZ-CZ", "CZ-PZ"]


def fetch(url, dst, tries=6):
    for k in range(tries):
        try:
            tmp = dst + ".part"
            with urllib.request.urlopen(url, timeout=180) as r, open(tmp, "wb") as f:
                while True:
                    b = r.read(1 << 20)
                    if not b:
                        break
                    f.write(b)
            os.replace(tmp, dst)
            return True
        except Exception as e:
            print(f"   retry {k+1}/{tries} {url}: {e}", flush=True)
            time.sleep(5 * (k + 1))
    return False


def read_edf(path):
    with open(path, "rb") as f:
        h = f.read(256)
        nrec = int(h[236:244]); dur = float(h[244:252]); ns = int(h[252:256])
        sh = f.read(256 * ns)
        nf = lambda start, width: [sh[start + i * width: start + (i + 1) * width].decode("latin-1").strip() for i in range(ns)]
        labels = nf(0, 16)
        p = 16 * ns + 80 * ns + 8 * ns
        pmin = np.array(nf(p, 8), float); p += 8 * ns
        pmax = np.array(nf(p, 8), float); p += 8 * ns
        dmin = np.array(nf(p, 8), float); p += 8 * ns
        dmax = np.array(nf(p, 8), float); p += 8 * ns
        p += 80 * ns
        nsamp = np.array(nf(p, 8), int)
        raw = np.frombuffer(f.read(), dtype="<i2")
    rec = int(nsamp.sum())
    nrec = min(nrec, raw.size // rec) if nrec > 0 else raw.size // rec
    raw = raw[: nrec * rec].reshape(nrec, rec)
    gain = (pmax - pmin) / (dmax - dmin)
    offs = np.concatenate([[0], np.cumsum(nsamp)])
    sigs = [((raw[:, offs[i]:offs[i + 1]].reshape(-1).astype(np.float32) - dmin[i]) * gain[i] + pmin[i]) for i in range(ns)]
    return labels, sigs, nsamp[0] / dur


def lowpass_decimate(x, fs):
    n = 129; t = np.arange(n) - (n - 1) / 2; fc = 50.0 / fs
    h = 2 * fc * np.sinc(2 * fc * t) * np.hamming(n); h /= h.sum()
    return np.convolve(x, h, mode="same")[::2]


def convert(edf, dst):
    labels, sigs, fs = read_edf(edf)
    up = [l.upper() for l in labels]
    out = []
    for ch in CHANNELS:
        idx = [i for i, l in enumerate(up) if l == ch or l.startswith(ch + "-")]
        if not idx:
            return f"missing channel {ch}"
        out.append(sigs[idx[0]])
    if abs(fs - 256) > 1:
        return f"unexpected fs {fs}"
    L = min(len(s) for s in out)
    X = np.stack([lowpass_decimate(s[:L], fs) for s in out]).astype(np.float16)
    np.savez_compressed(dst, x=X, fs=np.float32(fs / 2), ch=np.array(CHANNELS))
    return "ok"


def job(r, out, tmpdir):
    stem = os.path.basename(r)[:-4]
    dst = os.path.join(out, stem + ".npz")
    if os.path.exists(dst):
        return r, "exists"
    edf = os.path.join(tmpdir, os.path.basename(r))
    if not fetch(BASE + r, edf):
        return r, "download failed"
    try:
        st = convert(edf, dst)
    except Exception as e:
        st = f"error {e}"
    if os.path.exists(edf):
        os.remove(edf)
    return r, st


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--out", required=True)
    ap.add_argument("--patients", default=",".join(f"chb{i:02d}" for i in range(1, 11)))
    ap.add_argument("--n_free", type=int, default=3)
    ap.add_argument("--workers", type=int, default=6)
    a = ap.parse_args()
    os.makedirs(a.out, exist_ok=True)
    tmpdir = tempfile.mkdtemp(prefix="chbmit_edf_")
    for name in ["RECORDS", "RECORDS-WITH-SEIZURES", "SUBJECT-INFO"]:
        if not os.path.exists(os.path.join(a.out, name)):
            fetch(BASE + name, os.path.join(a.out, name))
    records = open(os.path.join(a.out, "RECORDS")).read().split()
    seiz = set(open(os.path.join(a.out, "RECORDS-WITH-SEIZURES")).read().split())
    todo = []
    for p in a.patients.split(","):
        s = os.path.join(a.out, f"{p}-summary.txt")
        if not os.path.exists(s):
            fetch(BASE + f"{p}/{p}-summary.txt", s)
        mine = [r for r in records if r.startswith(p + "/")]
        chosen = [r for r in mine if r in seiz] + [r for r in mine if r not in seiz][: a.n_free]
        print(f"{p}: {len(chosen)} recordings", flush=True)
        todo += chosen
    log = open(os.path.join(a.out, "log.txt"), "a")
    t0 = time.time()
    with ThreadPoolExecutor(a.workers) as ex:
        futs = [ex.submit(job, r, a.out, tmpdir) for r in todo]
        for k, fu in enumerate(as_completed(futs), 1):
            r, st = fu.result()
            log.write(f"{r}\t{st}\n"); log.flush()
            print(f"[{k}/{len(todo)}] {r}: {st}  ({(time.time()-t0)/60:.1f} min)", flush=True)
    print("Done. Folder:", a.out)


if __name__ == "__main__":
    main()

## 2. Download and prepare CHB-MIT (≈20–40 min the first time; skipped afterwards)

In [ ]:
!python -W ignore {CODE}/prepare_chbmit.py --out {PREP} --workers 6
!echo; echo "converted files:"; ls {PREP}/*.npz | wc -l; grep -v -c "ok\|exists" {PREP}/log.txt || true

## 3. Quick check (≈2 min): a short run on three patients to make sure everything works

In [ ]:
!python -W ignore {CODE}/run_loso.py --prep {PREP} --out /content/quickcheck --cache {CACHE} --models "EEGNet,GG-Net" --epochs 1 --patients chb01,chb02,chb03

## 4. Main comparison — all models, seed 0
Results for each model and held-out patient are written to `MyDrive/GhostGraph/results/runs/` as they finish.

In [ ]:
MAIN = "GG-Net,GGN (re-impl.),EEGNet,Static GCN,EEG Transformer,CNN-LSTM,ShallowConvNet,RF (handcrafted)"
!python -W ignore {CODE}/run_loso.py --prep {PREP} --out {OUT} --cache {CACHE} --models "{MAIN}" --epochs 15 --seeds 0

## 5. Ablation study — GG-Net with one component removed, seed 0

In [ ]:
ABL = "GG-Net w/o graph generator,GG-Net w/o Ghost (std conv),GG-Net w/o handcrafted,GG-Net w/o CNN branch"
!python -W ignore {CODE}/run_loso.py --prep {PREP} --out {OUT} --cache {CACHE} --models "{ABL}" --epochs 15 --seeds 0

## 6. Two extra training seeds for the main models (training-variance estimate; optional but recommended)

In [ ]:
!python -W ignore {CODE}/run_loso.py --prep {PREP} --out {OUT} --cache {CACHE} --models "{MAIN}" --epochs 15 --seeds 1,2

## 7. Tables, statistics, figures and the results package

In [ ]:
!python -W ignore {CODE}/analyze.py --runs {OUT}/runs --out {OUT}/analysis
!cp {PREP}/log.txt {OUT}/prep_log.txt; cp {PREP}/*-summary.txt {PREP}/SUBJECT-INFO {OUT}/ 2>/dev/null; true
!cd {ROOT} && rm -f ghostgraph_results.zip && zip -qr ghostgraph_results.zip results -x "results/cache/*" && ls -lh ghostgraph_results.zip
from IPython.display import Image, display
for f in ["fig2_roc.png", "fig4_per_patient.png", "fig3_graphs.png"]:
    p = f"{OUT}/analysis/{f}"
    if os.path.exists(p): display(Image(p, width=600))

## 8. Send the results back
Download **`MyDrive/GhostGraph/ghostgraph_results.zip`** (Files panel on the left → `drive/MyDrive/GhostGraph` → right-click → Download, or from drive.google.com) and attach it in the chat, or save it into your `eeg_dataset` folder. The manuscript tables, statistics and figures are then filled directly from these files.